In [1]:
import logging
import sys
from pathlib import Path
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import SparkSession

# Set up logging to show progress messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Find the project root folder dynamically
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Start local Spark session
spark = (SparkSession.builder
    .appName('MLlib-Model-Training')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '10')
    .getOrCreate())

print("✅ SparkSession successfully started!")

✅ SparkSession successfully started!


In [2]:
from pyspark.sql import functions as F

def train_saturation_model(aggregated_traffic_df):
    logger.info("Starting feature preparation with dynamic capacities...")
    
    # Define maximum capacity limits for different infrastructure types
    df_with_capacity = aggregated_traffic_df.withColumn(
        "max_capacity",
        F.when(F.col("infrastructure_type") == "bike_path", 3000.0)
         .when(F.col("infrastructure_type") == "pedestrian_zone", 600.0)
         .otherwise(1000.0)
    )
    
    # Calculate the saturation index label: count divided by max capacity
    df_with_label = df_with_capacity.withColumn(
        "saturation_index",
        F.col("count").cast("double") / F.col("max_capacity")
    )
    
    # Convert infrastructure text type into numbers so MLlib can use it
    indexer = StringIndexer(inputCol="infrastructure_type", outputCol="infra_type_encoded")
    df_indexed = indexer.fit(df_with_label).transform(df_with_label)
    
    # Combine features into a single vector column
    feature_cols = ["infra_type_encoded", "count"]
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    
    ml_data = (assembler
               .transform(df_indexed)
               .select("features", "saturation_index")
               .withColumnRenamed("saturation_index", "label"))
    
    # Split dataset into 70% training and 30% testing data
    logger.info("Splitting data into train and test sets...")
    train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=123)
    
    # Initialize and train the Linear Regression model
    logger.info("Training Advanced Linear Regression model...")
    lr = LinearRegression(featuresCol="features", labelCol="label", regParam=0.01)
    lr_model = lr.fit(train_data)
    
    return lr_model, test_data

In [3]:
def evaluate_model(lr_model, test_data):
    logger.info("Running predictions on test data...")
    predictions = lr_model.transform(test_data)
    
    # Set up evaluators to measure model performance
    evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
    
    # Calculate RMSE and R2 metrics
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)
    
    logger.info(f"Evaluation results -> RMSE: {rmse:.4f}, R²: {r2:.4f}")
    return predictions

In [4]:
def save_and_load_model(lr_model, path="models/saturation_linear_regression"):
    logger.info(f"Saving the trained model to {path}...")
    
    # Define the save directory path and export the model files
    full_path = PROJECT_ROOT / path
    lr_model.write().overwrite().save(str(full_path))

In [5]:
# Set path to the geospatial output data folder
simulation_path = PROJECT_ROOT / "data" / "geospatial_output"

# Read the simulated infrastructure counts from the CSV file
traffic_spatial_df = spark.read.csv(
    str(simulation_path / "infrastructure_activity_counts.csv"), 
    header=True, 
    inferSchema=True
)

# Run the machine learning workflow functions
lr_model, test_data = train_saturation_model(traffic_spatial_df)
predictions = evaluate_model(lr_model, test_data)

# Save the finished model locally
save_and_load_model(lr_model)

# Print out the first 5 prediction results
(predictions
 .select("features", "label", "prediction")
 .show(5, truncate=False))

2026-07-07 22:50:17,618 - INFO - Starting feature preparation with dynamic capacities...
2026-07-07 22:50:18,980 - INFO - Splitting data into train and test sets...
2026-07-07 22:50:18,992 - INFO - Training Advanced Linear Regression model...
2026-07-07 22:50:20,071 - INFO - Running predictions on test data...
2026-07-07 22:50:20,393 - INFO - Evaluation results -> RMSE: 0.0472, R²: 0.8953
2026-07-07 22:50:20,394 - INFO - Saving the trained model to models/saturation_linear_regression...


+-----------+------------------+------------------+
|features   |label             |prediction        |
+-----------+------------------+------------------+
|[0.0,461.0]|0.7683333333333333|0.7398064287417601|
|[0.0,500.0]|0.8333333333333334|0.7521640377186172|
|[0.0,500.0]|0.8333333333333334|0.7521640377186172|
|[1.0,500.0]|0.5               |0.4906295165043656|
|[1.0,500.0]|0.5               |0.4906295165043656|
+-----------+------------------+------------------+
only showing top 5 rows

